In [2]:
!uv pip install openai colorama streamlit beautifulsoup4

Using Python 3.12.3 environment at: /Users/shudhanshu/Desktop/Study Projects/GenAI/.venv
Resolved 50 packages in 267ms                                        
Prepared 14 packages in 13.93s                                           
Installed 14 packages in 66ms                               
 + altair==6.0.0
 + blinker==1.9.0
 + cachetools==7.0.5
 + colorama==0.4.6
 + gitdb==4.0.12
 + gitpython==3.1.46
 + narwhals==2.19.0
 + pandas==3.0.2
 + pillow==12.2.0
 + pyarrow==23.0.1
 + pydeck==0.9.1
 + smmap==5.0.3
 + streamlit==1.56.0
 + toml==0.10.2


In [3]:
!mkdir -p scratch_agent

In [6]:
%%writefile scratch_agent/__init__.py

print("init_file")

Writing scratch_agent/__init__.py


In [79]:
%%writefile scratch_agent/utils.py

"""
This is a collection of helper functions and methods we are going to use in the agent implementation. you dont need to know specific implementation
of these to follow the agent code. But, if you are curious feel free to check them out.
"""

import re
import time


from colorama import Fore
from colorama import Style

from dataclasses import dataclass

def completions_create(client, messages:list, model:str)->str:
    """
    Sends a request to the clients 'completion.create' method to interact with the language model.

    Args:
        client(OpenAI): The OpenAI client object
        messages(list[dict]): A List of messages object containing chat history for the model.
        model (str): The MOdel to use for generation tool calls and responses.

    Returns:-
        str: the contents of the models response
    """

    response = client.chat.completions.create(messages=messages, model=model)
    return str(response.choices[0].message.content)


def build_prompt_structure(prompt:str, role:str, tag:str="")->dict:
    """
    Builds a structured prompt that includes the role and content.

    Args:
        prompt(str): The actual content of the prompt.
        role (str): the role of the speaker (e.g:- user, assistant).

    Return:
    dict Adictonary representing the structured prompt.
    """

    if tag:
        prompt = f"<{tag}>{prompt}</{tag}>"
    return {"role": role, "content": prompt}

def update_chat_history(history:list,msg:str,role:str):
    """
    Updates the chat history by appending the latest response.

    Args:
        history(list): The List representing the current chat history.
        msg (str):The message to append.
        role(str): The role type (e.g. 'assistant', 'system')
    """

    history.append(build_prompt_structure(prompt=msg, role=role))



class ChatHistory(list):
    def __init__(self, messages:list|None=None, total_length:int=-1):
        """
        Initializes the queue with a fixed total length.

        Args:
            messages (list|None): Alist of initial messages
            total_length(int): The Maximum no of messages the chat history can hold.
        """

        if messages is None:
            messages = []

        super().__init__(messages)
        self.total_length = total_length


    def append(self, msg:str):
        """
        Add a message to the queue

        Args:
            msg(str): The message to be added to the queue
        """

        if len(self)==self.total_length:
            self.pop(0)
        super().append(msg)



class FixedFirstChatHistory(ChatHistory):
    def __init__(self, messages:list|None=None, total_length:int=-1):
        """Initialize the queue with a fixed total length.
    
            Args:
                messages(list|None): A list of Initial messages
                total_length (int): the maximum no of messages the chat history can hold.
        """
        super().__init__(messages, total_length)


    def append(self,msg:str):
        """Add a message to the queue. the first messages will always stay fixed.

            Args:
                msg(str): The message to be added to the queue
        """
        if len(self)==self.total_length:
            self.pop(1)
        super().aooend(msg)



def fancy_print(message:str)->None:
    """
    Display a fancy print message
    Args:
        message(str): The message to display.
        
    """

    print(Style.BRIGHT + Fore.CYAN +f"\n{'='*50}")
    print(Fore.MAGENTA + f"{message}")
    print(Style.BRIGHT + Fore.CYAN + f"{'='*50}\n")

    time.sleep(.5)


@dataclass
class TagContentResult:
    """
    A data class to represent the result of the extracting tag content.

    Attributes:
        content(List[str]): A list of strings containing the content found between the specified tags.
        found(bool): A flag indicating wheather any content was found for the give tag.
    """

    content: list[str]
    found: bool



def extract_tag_content(text:str, tag:str)->TagContentResult:
    """
    Extract all content enclosed by specific tags (e.g., '<thought>', '<response>', etc)

    Parameters:
        text(str): The input string containing multiple potential tags.
        tag(str): The name of the tag to search for (e.g., 'thought', 'response').

    Returns:
        dict: A dictionary with the following keys:
            -'content' (list): A list of strings cintaining the content found between the specified tags.
            -'found' (bool): A flag indicating wheather any content was found for the given tag
    
    """
    # Build the regex pattern dynamically to find the multipple occurence of the tag
    tag_pattern = rf"<{tag}>(.*?)</{tag}>"

    # use findall to capture all the content between the specified tag
    matched_contents = re.findall(tag_pattern, text, re.DOTALL)

    # return  the dataclass instance with the result
    return TagContentResult(
        content = [content.strip() for content in matched_contents],
        found = bool(matched_contents)
    )



Overwriting scratch_agent/utils.py


In [80]:
# Creating tools

In [81]:
%%writefile scratch_agent/tools.py

import os
import json
import re
from dataclasses import dataclass
from typing import Callable
from openai import OpenAI

# from google.colab import userdata

def get_fn_signature(fn:Callable)->dict:
    """
    Generates the signature for a given function

    Args:
        fn(Callable): The function whose signature needs to be extracted.

    Returns:
        dict: A dictionary containing the function's name, description, and parameter types.
    """
    fn_signature = {
        "name": fn.__name__,
        "description": fn.__doc__,
        "parameters":{"properties":{}}
    }

    schema = {
        k:{"type":v.__name__} for k,v in fn.__annotations__.items() if k!='return'
    }

    fn_signature['parameters']['properties'] = schema
    return fn_signature


def validate_arguments(tool_call:dict, tool_signature:dict)->dict:
    """
    Validates and converts argument in the input dict to match the expected types.

    Args:
        tool_call(dict): A dict containing the arguments passed to the tool.
        tool_signature(dict): The expected function signature and parameter types.

    Returns:
        dict: The tool call dict with arguments converted to the correct types if necessary.
        
    """

    properties = tool_signature['parameters']['properties']

    # TODO: This is overly simplified but enough for simple tools

    type_mapping = {
        'int':int,
        'str':str,
        'bool':bool,
        'float':float
    }

    for arg_name, arg_value in tool_call['arguments'].items():
        expected_type = properties[arg_name].get('type')

        if not isinstance(arg_value, type_mapping[expected_type]):
            tool_call['arguments'][arg_name] = type_mapping[expected_type](arg_value)

    return tool_call


class Tool:
    """
    A class representing a tool that wraps a callable and its signature.

    Attributes:
        name(str): The name of the tool(function)
        fn(Callable): The function that the tool represents.
        fn_signature(str): json String representing the fn signature.
    """

    def __init__(self, name:str, fn:Callable, fn_signature:str):
        self.name = name
        self.fn = fn
        self.fn_signature = fn_signature

    def __str__(self):
        return self.fn_signature

    def run(self, **kwargs):
        """
        Executes the tool(function) with provided arguments.

        Args:
            **kwargs: Keyword arguments passed to the function.

        Returns:
            The result of the function.
        """

        return self.fn(**kwargs)


def tool(fn:Callable):
    """
    A decorator that wraps a function into tool object.
    Args:
        fn(Callable): The function to be wrapped.

    Returns:
        Tool: A Tool object containing the function, its name, and signature.
        
    """

    def wrapper():
        fn_signature = get_fn_signature(fn)

        return Tool(
            name=fn_signature.get('name'), fn=fn, fn_signature=json.dumps(fn_signature)
        )
    return wrapper()

Overwriting scratch_agent/tools.py


In [82]:
# creating the react agent

In [83]:
%%writefile scratch_agent/react.py

import json
import re
from colorama import Fore
from openai import OpenAI


from scratch_agent.tools import tool, Tool, validate_arguments
from scratch_agent.utils import ChatHistory, completions_create, extract_tag_content, update_chat_history, build_prompt_structure

BASE_SYSTEM_PROMPT = ""

REACT_SYSTEM_PROMPT = """
    You operate by running a loop with the following steps: Thought Action, Observation.
    You are provided with function signature within <tools></tools> XML tags.
    You may call one or more functions to assist with the user query. Don't make assumptions about what value to plug
    into functions. Pay special attention to the properties 'types'. You should use those types as in a python dict.

    for each function call return a json object with function name and arguments within <tool_call></tool_call> XML tags as follow

    <tool_call>
    {'name':<function-name>,'arguments':<args-dict>,'id':<monotonically-increasing-id>}
    </tool_call>

    Here are the available tools / actions:

    <tools>
    %s
    </tools>

    Example Session:

    <question> whats the current temprature in delhi</question>
    <thought>I need to get the current weather in delhi</thought>
    <tool_call>{"name": "get_current_weather","arguments":{"location": "delhi", "unit": "celsius"}, "id":0}</tool_call>

    You will be called again with this:
    <observation>{0:{"temperature":25, "unit": "celsius"}}</observation>

    You then output:
    <response>The Current temperature in delhi is 25 degree Celsius</response>

    Additional constraints:

    -If the user asks you something unrelated to any of the tools above, answer freely enclosing your answer with <response></response> Tags.
    """



class ReactAgent:
    """
    A class that represents an agent uisng the ReAct logic that interacts with tools to process user inputs, make dicisions, and executes
     tool calls. the agent can run interactive sessions, collect tool signature, and process multiple tools calls in a given round of interaction.

     Attributes:
         client(OpenAI): The OpenAI client used to handle model-based completions.
         model(str): The name of the model used for generating responses. Default to 'GPT-4o'.
         tools(list[Tools]): A list of Tool instances available for execution.
         tools_dict: A dict mapping tool names to their corresponding Tool instances
    """

    def __init__(self, tools:Tool|list[Tool], model:str='gpt-4o', system_prompt:str=BASE_SYSTEM_PROMPT,api_key:str='')->None:
        
        self.client = OpenAI(api_key = api_key)
        self.model = model
        self.system_prompt = system_prompt

        self.tools = tools if isinstance(tools, list) else [tools]
        self.tools_dict = {tool.name: tool for tool in self.tools}

    def add_tool_signatures(self)->dict:
        """
        Collects the function signature of all available tools.

        Returns:
            str: A concatenated string of all tool function signature in JSON format.
        """
        return "".join([tool.fn_signature for tool in self.tools])


    def process_tool_calls(self, tool_calls_contents:list)->dict:
        """
        Processes each tool call validates arguments, executes the tools, and collects results.

        Args:
            tool_calls_content (list): List of strings each representing a tool call in json fromat.

        Returns:
            dict: A dictionary where the keys are tool call IDs and value are the results from the tools.
        """

        observations = {}
        for tool_call_str in tool_calls_contents:
            tool_call = json.loads(tool_call_str)
            tool_name = tool_call['name']
            tool = self.tools_dict[tool_name]

            print(Fore.GREEN +f"\nUsing Tool: {tool_name}")

            # Validate and  execute the tool call
            validated_tool_call = validate_arguments(
                tool_call, json.loads(tool.fn_signature)
            )

            print(Fore.GREEN + f"\nTool call dict: \n{validated_tool_call}")

            result = tool.run(**validated_tool_call['arguments'])
            print(Fore.GREEN + f"\nTool Result: \n{result}")

            # Store the result using the tool call ID
            observations[validated_tool_call['id']] = result

        return observations

    def run (self, user_msg:str, max_rounds:int=10)->str:
        """
        Executes a user interaction session where the agent processes user input, generates responses
        handles tool calls, and updates chat history until a final response is ready or the maximum number of rounds is reached.

        Args:
            user_msg(str): The users input message to start the interaction.
            max_rounds (int, optional): Maximum number of interaction rounds the agent should perform. default to 10.

        Returns:
            str: The final response generated by the agent after processing user input and any tool calls.
        """

        user_prompt = build_prompt_structure(
            prompt=user_msg, role='user', tags='question'
        )

        if self.tools:
            self.system_prompt+=("\n" + REACT_SYSTEM_PROMPT % self.add_tool_signatures())

        chat_history = ChatHistory([

            build_prompt_structure(
                prompt=self.system_prompt, role='system'
            ),
            user_prompt,
        ]
        )

        if self.tools:
        
            # Run the ReAct Loop for max_rounds

            for _ in range(max_rounds):
                completion = completions_create(self.client, chat_history, self.model)

                response = extract_tag_content(str(completion), "response")

                if response.found:
                    return response.content[0]

                thought = extract_tag_content(str(completion),'thought')
                tool_calls = extract_tag_content(str(completion),'tool_call')

                update_chat_history(chat_history, completion, 'assistant')

                print(Fore.MAGENTA + f"\nThought: {thought.content[0]}")


                if tool_calls.found:
                    observations = self.process_tools_calls(tool_calls.content)
                    print(Fore.BLUE + f"\bObservations: {observations}")
                    update_chat_history(chat_history, f"{observations}", "user")

        return completions_create(self.client, chat_history, self.model)
                    
        

        
        
        








Overwriting scratch_agent/react.py


In [84]:
# Defining the tools

In [85]:
%%writefile tools.py
import json
import requests
from bs4 import BeautifulSoup
from scratch_agent.tools import tool

BASE_URL = "https://hacker-news.firebaseio.com/v0"

def fetch_item(item_id:int):
    """
    Fetches details of a story by its ID.
    Args:
        item_id(int): The Id of the item to fetch.
    Returns:
        dict: Details of the story.
    """

    url = f"{BASE_URL}/item/{item_id}.json"
    response = requests.get(url)

    return response.json()


def fetch_story_ids(story_type:str='top', limit:int=None):
    """
    Fetches the top story IDs.

    Args:
        story_type: the story type. Defaults to top('topstories.json')
        limit: the limit of stories to be fetched.

    Returns:
        List[int]: A list of top story IDs

    """
    url = f"{BASE_URL}/{story_type}storeis.json"
    response = requests.get(url)

    story_ids = response.json()

    if limit:
        story_ids = story_ids[:limit]

    return story_ids


def fetch_text(url:str):
    """
    Fetches the text from a url (if theres text to be fetched). if it fails,
    it will return an informative message to the llm

    Args:
        url: the story url.
    Returns:
        A string representing whether the story text ir an informative error(represented as string)
    """

    try:
        response = requests.get(url)

        if response.status_code==200:

            html_content = response.content
            soup = BeautifulSoup(html_content, 'html.parser')
            text_content = soup.get_text()

            return text_content
        else:
            return f"Unable to fetch content from {url}. Status code ={response.status_code} "
    except Exception as e:
        return f"An error occured: {e}"





@tool
def get_hn_stories(limit: int=5, story_type: str="top"):
    """
    Fetches the top Hacker News Stories Based on the provided parameters.

    Args:
        limit(int): The number of the top stories to retrieve. Default is 10.
        keywords(List[str]): A list of keywords to filter the top storeis.
        story_type(str): The story type

    Returns:
        list[Dict[str,union[str,int]]]: A list of dict containing story_id, title, url and score of the stories.
    """

    if limit:
        story_id = fetch_story_ids(story_type, limit)

    else:
        story_ids = fetch_story_ids(story_type)

    def fetch_and_filter_stories(story_id):
        return fetch_item(story_id)

    stories = [fetch_and_filter_stories(story_id) for story_id in story_ids]
    formatted_stories = []

    for story in stories:
        story_info = {
            "title":story.get('title'),
             "url":story.get('url'),
             "score":story.get('score'),
             "story_id":story.get('story_id'),
        }
        formatted_stories.append(story_info)

    return formatted_stories[:limit]



@tool
def get_relevant_comments(story_id: int, limit: int=10):
    """
    Get the most relevant comments for a hacker news item

    Args:
        story_id: the id of the hn item
        limit: no of commentsbto retrieve (default to 10)

    Returns:
        A list of dict, each containing comment details
    """
    story =fetch_item(story_id)

    if 'kids' not in story:
        return "this item does not have comments"

    comment_ids = story['kids']

    comment_details = [fetch_item(cid) for cid in comment_ids]
    comment_details.sort(key=lambda coment: coment.get('score', 0), revese=True)

    relevent_comments = comment_details[:limit]
    relevent_comments = [comment['text'] for comment in relevent_comments]

    return json.dumps(relevent_comments)


@tool
def get_story_content(story_url:str):
    """
    gets the content of the story
    Args:
        story_url: url of story
    Returns:
        the content of story
    """
    return fetch_text(story_url)
    

Overwriting tools.py


In [90]:
%%writefile hn_bot.py

import os
import re
import math
import json
# from google.colab import user_data

from openai import OpenAI

from scratch_agent.react import ReactAgent
from tools import get_hn_stories, get_relevant_comments, get_story_content

def get_hb_bot(api_key:str):
    bot_system_prompt = """ You are the Singularity Incarnation of Hacker News.
    The human will ask you for information about hacker news. If you cant find any information
    about the question asked or the result is incomplete, apologies to the human and ask him if 
    you can help him with something else.

    If the human asks you to show him stories, do it using markdown tables.
    The markdown tables has the following format:

    story_id|title|url|score"""

    agent = ReactAgent(
        system_prompt = bot_system_prompt,
        tools =[get_hn_stories, get_relevant_comments, get_story_content],
        api_key = api_key
        
    )
    return agent






Overwriting hn_bot.py


In [91]:
%%writefile app.py

import os
import asyncio

from PIL import Images
import streamlit as st

from hn_bot import get_hb_bot

# set streamlit pageconfig

st.set_page_config(page_title="Scratch Agent implementation")
st.title("Scratch Agent implementation")

with st.sidebar:
    st.markdown(
        """
        # **Greetings, Digital Explorer!**

        Are you fatigued from navigating the expansive digital realm in search of daily tech tales.
        Look No-Where else.
        """)

    api_key = st.text_input("Enter the key:", type ="password")
    st.session_state["agent"] = get_hb_bot(api_key=api_key)


if "messages" not in st.session_state:
    st.session_state["messages"] = []

def generate_response(question):
    context = "\n".join([msg['bot'] for msg in st.session_state['messsages']])
    response = st.session_state["agent"].run(f'context:{context} Question:{question}')

    return response

# display chat history
for msg in st.session_state["messages"]:
    st.chat_message("human").write(msg['user'])
    st.chat_message("ai").write(msg['bot'])


# chat input handling
if prompt:=st.chat_input():
    st.chat_message('human').write(prompt)

    with st.spinner('thinking....'):
        response = generate_response(prompt)

        st.chat_message('ai').write(response)

    # store history
    st.session_state["mesage"].append({'user':prompt,'bot':response})
    
    
    
    
    

Writing app.py


In [92]:

!npm install localtunnel

m##################) ⠙ reify:yargs: http fetch GET 200 https://registry.npmjs.opmjs.o
added 22 packages in 3s

3 packages are looking for funding
  run `npm fund` for details
npm notice 
npm notice New major version of npm available! 10.2.3 -> 11.12.1
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.12.1
npm notice Run npm install -g npm@11.12.1 to update!
npm notice 


In [93]:
!streamlit run app.py --server.address=localhost &>/content/logs.txt &

OSError: Background processes not supported.